# Step 8 Select GenAI Labeling Strategy and Label Train/Test

This notebook moves the model-comparison workflow out of Step 7. Step 7 generates model labels; Step 8 compares those labels against the Step 6 human consensus, selects a GenAI labeling strategy, and labels the Step 5 train/test files with the selected model.

Run this notebook after these files exist in Google Drive:

- `Team 8 - Capstone Project/Step 5 - Preprocess Split/data/train.csv`
- `Team 8 - Capstone Project/Step 5 - Preprocess Split/data/test.csv`
- `Team 8 - Capstone Project/Step 5 - Preprocess Split/data/train_test_15000.csv`
- `Team 8 - Capstone Project/Step 6 - Human Labeling/data/holdout_with_consensus.csv`
- `Team 8 - Capstone Project/Step 7 - GenAI Benchmark/data/*_labels.csv`
- `Team 8 - Capstone Project/Step 7 - GenAI Benchmark/benchmark_summary.csv`

Generated Step 8 outputs are written to:

- `Team 8 - Capstone Project/Step 8 - Select GenAI Labeling Strategy/data/step8_benchmark_results.csv`
- `Team 8 - Capstone Project/Step 8 - Select GenAI Labeling Strategy/data/selected_labeling_strategy.txt`
- `Team 8 - Capstone Project/Step 8 - Select GenAI Labeling Strategy/data/confusion_matrix_*.png`
- `Team 8 - Capstone Project/Step 8 - Select GenAI Labeling Strategy/data/train_labeled_*.csv`
- `Team 8 - Capstone Project/Step 8 - Select GenAI Labeling Strategy/data/test_labeled_*.csv`
- `Team 8 - Capstone Project/Step 8 - Select GenAI Labeling Strategy/data/train_test_labeled_*.csv`

## 1. Install Dependencies, Mount Drive, and Configure Paths

This section installs notebook dependencies, mounts Google Drive, and points to the Step 5, Step 6, Step 7, and Step 8 Drive folders.

In [ ]:
# Install dependencies in a fresh Colab runtime.
%pip install -q anthropic google-generativeai krippendorff matplotlib openai pandas python-dotenv scikit-learn seaborn tqdm

import asyncio
import logging
import os
import random
import re
import time
import warnings
from dataclasses import dataclass
from importlib import import_module
from pathlib import Path
from typing import Awaitable, Callable

import krippendorff
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from anthropic import AsyncAnthropic
from dotenv import load_dotenv
from openai import AsyncOpenAI
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
)
from tqdm.auto import tqdm

# Mount Google Drive so inputs and outputs persist for everyone.
try:
    drive = import_module("google.colab").drive
except ModuleNotFoundError as exc:
    raise RuntimeError("Run this notebook in Google Colab so Google Drive can be mounted.") from exc

drive.mount("/content/drive")

PROJECT_DRIVE_BASE = Path("/content/drive/MyDrive/Team 8 - Capstone Project")
STEP5_DRIVE_BASE = PROJECT_DRIVE_BASE / "Step 5 - Preprocess Split"
STEP6_DRIVE_BASE = PROJECT_DRIVE_BASE / "Step 6 - Human Labeling"
STEP7_DRIVE_BASE = PROJECT_DRIVE_BASE / "Step 7 - GenAI Benchmark"
STEP8_DRIVE_BASE = PROJECT_DRIVE_BASE / "Step 8 - Select GenAI Labeling Strategy"

STEP5_DATA_DIR = STEP5_DRIVE_BASE / "data"
STEP6_DATA_DIR = STEP6_DRIVE_BASE / "data"
STEP7_DATA_DIR = STEP7_DRIVE_BASE / "data"
STEP8_DATA_DIR = STEP8_DRIVE_BASE / "data"

HUMAN_CONSENSUS_FILE = STEP6_DATA_DIR / "holdout_with_consensus.csv"
BENCHMARK_SUMMARY_FILE = STEP7_DRIVE_BASE / "benchmark_summary.csv"
TRAIN_FILE = STEP5_DATA_DIR / "train.csv"
TEST_FILE = STEP5_DATA_DIR / "test.csv"
TRAIN_TEST_FILE = STEP5_DATA_DIR / "train_test_15000.csv"

BENCHMARK_RESULTS_FILE = STEP8_DATA_DIR / "step8_benchmark_results.csv"
SELECTED_STRATEGY_FILE = STEP8_DATA_DIR / "selected_labeling_strategy.txt"
LOG_FILE = STEP8_DRIVE_BASE / "step8_label_train_test.log"

STEP8_DATA_DIR.mkdir(parents=True, exist_ok=True)
STEP8_DRIVE_BASE.mkdir(parents=True, exist_ok=True)

print(f"Human consensus input: {HUMAN_CONSENSUS_FILE}")
print(f"Step 7 model labels: {STEP7_DATA_DIR}")
print(f"Step 8 output directory: {STEP8_DATA_DIR}")

## 2. Compare Step 7 Model Labels Against Human Consensus

This section evaluates each Step 7 model against the Step 6 human consensus. It computes accuracy, Krippendorff's alpha, Cohen's kappa, MCC, macro/weighted F1, per-class F1, and confusion matrices.

In [ ]:
HUMAN_TO_MODEL_LABEL = {
    "neither": "NEITHER",
    "exploitation": "EXPLOITATION",
    "exploration": "EXPLORATION",
    "ambiguous": "AMBIDEXTROUS",
    "ambidextrous": "AMBIDEXTROUS",
}
MODEL_LABELS = ["NEITHER", "EXPLOITATION", "EXPLORATION", "AMBIDEXTROUS"]
LABEL_TO_CODE = {label: index for index, label in enumerate(MODEL_LABELS)}


def verify_evaluation_inputs():
    missing = []
    for path in [HUMAN_CONSENSUS_FILE, BENCHMARK_SUMMARY_FILE, TRAIN_FILE, TEST_FILE, TRAIN_TEST_FILE]:
        if not path.exists():
            missing.append(str(path))
    if not STEP7_DATA_DIR.exists() or not list(STEP7_DATA_DIR.glob("*_labels.csv")):
        missing.append(str(STEP7_DATA_DIR / "*_labels.csv"))

    if missing:
        raise FileNotFoundError("Missing required Step 8 inputs:\n" + "\n".join(missing))

    test_file = STEP8_DATA_DIR / "test_file.csv"
    pd.DataFrame({"check": [1, 2, 3]}).to_csv(test_file, index=False)
    print(f"Step 8 Drive write test succeeded: {test_file}")


def normalize_human_label(label):
    normalized = str(label).strip().lower()
    return HUMAN_TO_MODEL_LABEL.get(normalized, "INVALID")


def load_human_consensus():
    df = pd.read_csv(HUMAN_CONSENSUS_FILE, dtype=str, encoding="utf-8-sig").fillna("")
    required = {"sentence_id", "consensus"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Human consensus file is missing required columns: {missing}")

    df["human_label"] = df["consensus"].map(normalize_human_label)
    invalid = sorted(set(df["human_label"]) - set(MODEL_LABELS))
    if invalid:
        raise ValueError(f"Unexpected human labels after normalization: {invalid}")

    return df[["sentence_id", "human_label"]]


def load_model_labels(label_file):
    df = pd.read_csv(label_file, dtype=str, encoding="utf-8-sig").fillna("")
    required = {"sentence_id", "label", "model"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"{label_file} is missing required columns: {missing}")

    df["label"] = df["label"].str.strip().str.upper()
    return df


def krippendorff_alpha_two_raters(human_labels, model_labels):
    coded = [
        [LABEL_TO_CODE[label] for label in human_labels],
        [LABEL_TO_CODE[label] for label in model_labels],
    ]
    return krippendorff.alpha(reliability_data=coded, level_of_measurement="nominal")


def save_confusion_matrix(model_name, y_true, y_pred):
    matrix = confusion_matrix(y_true, y_pred, labels=MODEL_LABELS)
    matrix_df = pd.DataFrame(matrix, index=MODEL_LABELS, columns=MODEL_LABELS)

    plt.figure(figsize=(7, 6))
    sns.heatmap(matrix_df, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{model_name} vs Human Consensus")
    plt.ylabel("Human consensus")
    plt.xlabel("Model label")
    plt.tight_layout()

    safe_model = model_name.replace(".", "_").replace("/", "_")
    output_path = STEP8_DATA_DIR / f"confusion_matrix_{safe_model}.png"
    plt.savefig(output_path, dpi=200)
    plt.close()
    return output_path


def evaluate_model(label_file, human_df):
    model_df = load_model_labels(label_file)
    model_name = model_df["model"].replace("", pd.NA).dropna().iloc[0]
    merged = human_df.merge(model_df, on="sentence_id", how="inner")
    merged = merged[merged["label"].isin(MODEL_LABELS)].copy()

    y_true = merged["human_label"]
    y_pred = merged["label"]
    report = classification_report(
        y_true,
        y_pred,
        labels=MODEL_LABELS,
        output_dict=True,
        zero_division=0,
    )

    alpha = krippendorff_alpha_two_raters(y_true.tolist(), y_pred.tolist())
    confusion_path = save_confusion_matrix(model_name, y_true, y_pred)

    return {
        "model": model_name,
        "n_compared": len(merged),
        "accuracy": accuracy_score(y_true, y_pred),
        "krippendorff_alpha": alpha,
        "cohen_kappa": cohen_kappa_score(y_true, y_pred, labels=MODEL_LABELS),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=MODEL_LABELS, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=MODEL_LABELS, average="weighted", zero_division=0),
        "f1_neither": report["NEITHER"]["f1-score"],
        "f1_exploitation": report["EXPLOITATION"]["f1-score"],
        "f1_exploration": report["EXPLORATION"]["f1-score"],
        "f1_ambiguous": report["AMBIDEXTROUS"]["f1-score"],
        "confusion_matrix_file": str(confusion_path),
    }


def build_benchmark_results():
    verify_evaluation_inputs()
    human_df = load_human_consensus()
    summary_df = pd.read_csv(BENCHMARK_SUMMARY_FILE, encoding="utf-8-sig")

    rows = []
    for label_file in sorted(STEP7_DATA_DIR.glob("*_labels.csv")):
        rows.append(evaluate_model(label_file, human_df))

    results_df = pd.DataFrame(rows)
    results_df = results_df.merge(summary_df, on="model", how="left")
    results_df["alpha_per_dollar"] = results_df.apply(
        lambda row: row["krippendorff_alpha"] / row["estimated_cost_usd"]
        if pd.notna(row.get("estimated_cost_usd")) and row["estimated_cost_usd"] > 0
        else pd.NA,
        axis=1,
    )
    results_df = results_df.sort_values(
        ["macro_f1", "krippendorff_alpha", "mcc"],
        ascending=False,
    )
    results_df.to_csv(BENCHMARK_RESULTS_FILE, index=False)
    return results_df


benchmark_results = build_benchmark_results()
print(f"Saved benchmark results to: {BENCHMARK_RESULTS_FILE}")
benchmark_results

## 3. Select Labeling Strategy

Choose the model using a metric. The default chooses the highest `macro_f1`, with Krippendorff's alpha and MCC as tie-breakers.

In [ ]:
# Change this if your selection criterion changes.
SELECTION_METRIC = "macro_f1"

selected_row = benchmark_results.sort_values(
    [SELECTION_METRIC, "krippendorff_alpha", "mcc"],
    ascending=False,
).iloc[0]
SELECTED_MODEL = selected_row["model"]

strategy_text = f"""Selected GenAI labeling strategy
================================
Selected model: {SELECTED_MODEL}
Selection metric: {SELECTION_METRIC}
{SELECTION_METRIC}: {selected_row[SELECTION_METRIC]}
Krippendorff alpha: {selected_row['krippendorff_alpha']}
Cohen kappa: {selected_row['cohen_kappa']}
MCC: {selected_row['mcc']}
Macro F1: {selected_row['macro_f1']}
Estimated cost USD on holdout: {selected_row.get('estimated_cost_usd', 'n/a')}
"""

SELECTED_STRATEGY_FILE.write_text(strategy_text, encoding="utf-8")
print(strategy_text)
print(f"Saved selected strategy to: {SELECTED_STRATEGY_FILE}")

## 4. API Keys for Train/Test Labeling

Set the API key for the selected model provider. You can put a `.env` file in the Step 8 Drive folder or paste keys into this runtime cell.

In [ ]:
# Option 1: Put a .env file in the Step 8 Drive folder with keys like:
# OPENAI_API_KEY=...
# ANTHROPIC_API_KEY=...
# GOOGLE_API_KEY=...
load_dotenv(STEP8_DRIVE_BASE / ".env")

# Option 2: Paste keys here for the current Colab runtime.
OPENAI_API_KEY = ""
ANTHROPIC_API_KEY = ""
GOOGLE_API_KEY = ""

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if ANTHROPIC_API_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("API key availability")
print(f"OpenAI: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"Anthropic: {bool(os.getenv('ANTHROPIC_API_KEY'))}")
print(f"Google: {bool(os.getenv('GOOGLE_API_KEY'))}")

## 5. Label Train/Test with Selected Model

This section uses the selected model to label Step 5's train, test, and combined train/test files. Outputs checkpoint after each batch and can resume if Colab disconnects.

In [ ]:
SYSTEM_PROMPT = """You are a research assistant classifying sentences from corporate 10-K annual report filings.

For each sentence, assign exactly one of these four labels:

EXPLORATION — the firm describes novelty, experimentation, new markets, new capabilities, R&D, pilots, or building something it does not yet possess. The outcome is uncertain. The capability or knowledge is genuinely new to the firm.

EXPLOITATION — the firm describes improving, scaling, optimizing, standardizing, or extracting more value from existing operations, products, or capabilities. The outcome is predictable. The capability already exists within the firm.

AMBIDEXTROUS — both EXPLORATION and EXPLOITATION are clearly and equally present in the same sentence. Both must be explicitly stated and structurally parallel. If one is secondary, assign the dominant label instead.

NEITHER — financial figures, legal boilerplate, risk disclosures, generic aspirational language, or any sentence with no clear strategic orientation.

Respond with ONLY the label. No explanation. No punctuation. Just one word: EXPLORATION, EXPLOITATION, AMBIDEXTROUS, or NEITHER."""

VALID_LABELS = {"EXPLORATION", "EXPLOITATION", "AMBIDEXTROUS", "NEITHER"}
LABEL_OUTPUT_COLUMNS = ["sentence_id", "label", "model", "latency_seconds"]
BATCH_SIZE = 50
MAX_RETRIES = 8
MAX_OUTPUT_TOKENS = 256


@dataclass(frozen=True)
class ModelConfig:
    name: str
    provider: str
    input_cost_per_1k_tokens: float
    output_cost_per_1k_tokens: float
    estimated_cost_per_1000_sentences: float = 0.0
    max_concurrency: int = 2
    min_request_interval_seconds: float = 0.5


@dataclass
class LabelResult:
    sentence_id: str
    label: str
    model: str
    latency_seconds: float
    input_tokens: int = 0
    output_tokens: int = 0


MODEL_CONFIGS = {
    "gpt-4o-mini": ModelConfig("gpt-4o-mini", "openai", 0.00015, 0.0006, 0.05, 5, 0.2),
    "gpt-4o": ModelConfig("gpt-4o", "openai", 0.005, 0.015, 2.50, 1, 1.2),
    "gemini-2.5-flash": ModelConfig("gemini-2.5-flash", "google", 0.00015, 0.00060),
    "gemini-2.5-flash-lite": ModelConfig("gemini-2.5-flash-lite", "google", 0.000075, 0.0003),
    "claude-haiku-4-5-20251001": ModelConfig("claude-haiku-4-5-20251001", "anthropic", 0.00025, 0.00125, 0.15, 1, 1.4),
    "claude-sonnet-4-6": ModelConfig("claude-sonnet-4-6", "anthropic", 0.003, 0.015, 1.50, 1, 1.4),
}


def normalize_model_label(raw_label, model_name, sentence_id):
    label = raw_label.strip().upper()
    if label in VALID_LABELS:
        return label

    logging.warning("Invalid label from %s for sentence_id=%s: %r", model_name, sentence_id, raw_label)
    return "INVALID"


def sentence_prompt(sentence):
    return f"Classify this sentence:\n\n{sentence}"


def is_rate_limit_error(error):
    error_name = error.__class__.__name__.lower()
    error_message = str(error).lower()
    retry_terms = ["ratelimit", "rate limit", "429", "quota", "resource_exhausted"]
    return any(term in error_name or term in error_message for term in retry_terms)


def retry_delay_seconds(error, attempt):
    error_message = str(error).lower()
    milliseconds_match = re.search(r"try again in ([0-9.]+)ms", error_message)
    if milliseconds_match:
        return max(2.0, float(milliseconds_match.group(1)) / 1000)
    seconds_match = re.search(r"try again in ([0-9.]+)s", error_message)
    if seconds_match:
        return max(2.0, float(seconds_match.group(1)))
    return min(60.0, (2**attempt) + random.uniform(0, 0.5))


class RequestPacer:
    def __init__(self, min_interval_seconds):
        self.min_interval_seconds = min_interval_seconds
        self.last_request_time = 0.0
        self.lock = asyncio.Lock()

    async def wait(self):
        if self.min_interval_seconds <= 0:
            return
        async with self.lock:
            elapsed = time.perf_counter() - self.last_request_time
            if elapsed < self.min_interval_seconds:
                await asyncio.sleep(self.min_interval_seconds - elapsed)
            self.last_request_time = time.perf_counter()


async def retry_rate_limits(call: Callable[[], Awaitable[LabelResult]], model_name, sentence_id):
    for attempt in range(MAX_RETRIES + 1):
        try:
            return await call()
        except Exception as error:
            if not is_rate_limit_error(error) or attempt == MAX_RETRIES:
                raise
            sleep_seconds = retry_delay_seconds(error, attempt)
            logging.warning("Rate limit for %s sentence_id=%s. Retry %s/%s in %.2fs.", model_name, sentence_id, attempt + 1, MAX_RETRIES, sleep_seconds)
            await asyncio.sleep(sleep_seconds)


async def call_openai(client, config, row):
    start_time = time.perf_counter()
    response = await client.chat.completions.create(
        model=config.name,
        temperature=0,
        max_tokens=MAX_OUTPUT_TOKENS,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": sentence_prompt(row["sentence"])},
        ],
    )
    latency = time.perf_counter() - start_time
    text = response.choices[0].message.content if response.choices else ""
    usage = response.usage
    return LabelResult(row["sentence_id"], normalize_model_label(text or "", config.name, row["sentence_id"]), config.name, latency, getattr(usage, "prompt_tokens", 0), getattr(usage, "completion_tokens", 0))


async def call_anthropic(client, config, row):
    start_time = time.perf_counter()
    response = await client.messages.create(
        model=config.name,
        max_tokens=MAX_OUTPUT_TOKENS,
        temperature=0,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": sentence_prompt(row["sentence"])}],
    )
    latency = time.perf_counter() - start_time
    text = response.content[0].text if response.content else ""
    usage = response.usage
    return LabelResult(row["sentence_id"], normalize_model_label(text, config.name, row["sentence_id"]), config.name, latency, getattr(usage, "input_tokens", 0), getattr(usage, "output_tokens", 0))


async def call_google(model, genai, config, row):
    prompt = f"{SYSTEM_PROMPT}\n\n{sentence_prompt(row['sentence'])}"
    start_time = time.perf_counter()
    response = await model.generate_content_async(
        prompt,
        generation_config=genai.types.GenerationConfig(temperature=0, max_output_tokens=MAX_OUTPUT_TOKENS),
    )
    latency = time.perf_counter() - start_time
    usage = getattr(response, "usage_metadata", None)
    try:
        text = response.text or ""
    except ValueError:
        text = ""
    return LabelResult(row["sentence_id"], normalize_model_label(text, config.name, row["sentence_id"]), config.name, latency, getattr(usage, "prompt_token_count", 0), getattr(usage, "candidates_token_count", 0))


def build_model_client(config):
    if config.provider == "openai":
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY is required for the selected OpenAI model.")
        client = AsyncOpenAI(api_key=api_key)
        return lambda row: call_openai(client, config, row)

    if config.provider == "anthropic":
        api_key = os.getenv("ANTHROPIC_API_KEY")
        if not api_key:
            raise ValueError("ANTHROPIC_API_KEY is required for the selected Anthropic model.")
        client = AsyncAnthropic(api_key=api_key)
        return lambda row: call_anthropic(client, config, row)

    if config.provider == "google":
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=FutureWarning)
            import google.generativeai as genai
        api_key = os.getenv("GOOGLE_API_KEY")
        if not api_key:
            raise ValueError("GOOGLE_API_KEY is required for the selected Google model.")
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel(config.name)
        return lambda row: call_google(model, genai, config, row)

    raise ValueError(f"Unknown provider: {config.provider}")


def label_results_to_dataframe(results):
    return pd.DataFrame(
        [{"sentence_id": result.sentence_id, "label": result.label, "model": result.model, "latency_seconds": round(result.latency_seconds, 4)} for result in results],
        columns=LABEL_OUTPUT_COLUMNS,
    )


async def label_dataset(input_file, output_file, config):
    source_df = pd.read_csv(input_file, dtype=str, encoding="utf-8-sig").fillna("")
    existing_labels = pd.DataFrame(columns=LABEL_OUTPUT_COLUMNS)
    processed_ids = set()

    if output_file.exists():
        existing_labels = pd.read_csv(output_file, dtype={"sentence_id": str}, encoding="utf-8-sig")
        processed_ids = set(existing_labels["sentence_id"].astype(str))
        print(f"Resuming {output_file.name}: {len(processed_ids)}/{len(source_df)} labels already saved.")

    remaining_df = source_df[~source_df["sentence_id"].astype(str).isin(processed_ids)]
    call_model = build_model_client(config)
    semaphore = asyncio.Semaphore(config.max_concurrency)
    pacer = RequestPacer(config.min_request_interval_seconds)
    new_results = []

    async def label_one(row):
        async def paced_call():
            await pacer.wait()
            return await call_model(row)
        async with semaphore:
            return await retry_rate_limits(paced_call, config.name, row["sentence_id"])

    with tqdm(total=len(source_df), initial=len(processed_ids), desc=output_file.name, unit="sentence") as progress:
        for batch_start in range(0, len(remaining_df), BATCH_SIZE):
            batch = remaining_df.iloc[batch_start : batch_start + BATCH_SIZE]
            batch_results = await asyncio.gather(*[label_one(row) for row in batch.to_dict(orient="records")])
            new_results.extend(batch_results)
            checkpoint = pd.concat([existing_labels, label_results_to_dataframe(new_results)], ignore_index=True)
            checkpoint.to_csv(output_file, index=False)
            progress.update(len(batch_results))

    final_labels = pd.concat([existing_labels, label_results_to_dataframe(new_results)], ignore_index=True)
    final_labels.to_csv(output_file, index=False)

    labeled_df = source_df.merge(final_labels[["sentence_id", "label", "model"]], on="sentence_id", how="left")
    labeled_df = labeled_df.rename(columns={"label": "genai_label", "model": "genai_model"})
    labeled_df.to_csv(output_file.with_name(output_file.stem.replace("_labels", "") + ".csv"), index=False)
    return labeled_df


logging.basicConfig(filename=LOG_FILE, level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s")

In [ ]:
# Set RUN_TRAIN_TEST_LABELING to True when you are ready to label the full train/test data.
# This may take time and incur API cost.
RUN_TRAIN_TEST_LABELING = False

selected_config = MODEL_CONFIGS[SELECTED_MODEL]
safe_model_name = SELECTED_MODEL.replace(".", "_").replace("/", "_")

TRAIN_LABELS_FILE = STEP8_DATA_DIR / f"train_labeled_{safe_model_name}_labels.csv"
TEST_LABELS_FILE = STEP8_DATA_DIR / f"test_labeled_{safe_model_name}_labels.csv"
TRAIN_TEST_LABELS_FILE = STEP8_DATA_DIR / f"train_test_labeled_{safe_model_name}_labels.csv"

if RUN_TRAIN_TEST_LABELING:
    train_labeled = await label_dataset(TRAIN_FILE, TRAIN_LABELS_FILE, selected_config)
    test_labeled = await label_dataset(TEST_FILE, TEST_LABELS_FILE, selected_config)
    train_test_labeled = await label_dataset(TRAIN_TEST_FILE, TRAIN_TEST_LABELS_FILE, selected_config)
    print("Finished labeling train, test, and train/test files.")
else:
    print("Train/test labeling is currently disabled.")
    print("Set RUN_TRAIN_TEST_LABELING = True when you are ready to run paid model labeling.")
    print(f"Selected model: {SELECTED_MODEL}")

## 6. Output Checks

These checks show the Step 8 outputs that have been generated so far.

In [ ]:
print(f"Benchmark results: {BENCHMARK_RESULTS_FILE.exists()} -> {BENCHMARK_RESULTS_FILE}")
print(f"Selected strategy: {SELECTED_STRATEGY_FILE.exists()} -> {SELECTED_STRATEGY_FILE}")
print(f"Step 8 log: {LOG_FILE.exists()} -> {LOG_FILE}")

print("\nConfusion matrices:")
for path in sorted(STEP8_DATA_DIR.glob("confusion_matrix_*.png")):
    print(f"- {path.name}")

print("\nLabeled train/test outputs:")
for path in sorted(STEP8_DATA_DIR.glob("*_labeled_*.csv")):
    print(f"- {path.name}")

pd.read_csv(BENCHMARK_RESULTS_FILE).head(10)

## Notes on Reproducibility and Limitations

- Step 7 now remains responsible for generating benchmark model labels.
- Step 8 is responsible for comparing model labels to human consensus, selecting a model, and applying that model to train/test.
- Human `ambiguous` consensus labels are mapped to model `AMBIDEXTROUS` labels for evaluation because the Step 7 prompt uses `AMBIDEXTROUS`.
- Train/test labeling is disabled by default because it can take time and incur API cost. Set `RUN_TRAIN_TEST_LABELING = True` when ready.
- Train/test labeling checkpoints after every batch and resumes from existing label CSVs if Colab disconnects.